# Hansen Ch.21 Regression Discontinuity

理论见 `Hansen_Ch21_Exercises_Solutions.md`（**21.1–21.9**）。

数据：`LM2007`（Head Start）。代码含中文注释。

In [ ]:

# Hansen Ch.21 RDD utilities — 详尽注释
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import pinv

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

# Head Start 门槛：约 300 个最穷县对应的 1960 贫困率
C_CUTOFF = 59.1984


def load_lm2007():
    """读取 Ludwig-Miller / Cattaneo 整理的县一级数据。"""
    df = pd.read_excel(ROOT / "LM2007/LM2007.xlsx")
    for col in df.columns:
        if col != "state":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def triangular_normalized_kernel(u):
    """Table 19.1 标准化三角核：支撑 |u|<sqrt(6)，int u^2 K=1。"""
    a = np.sqrt(6.0)
    return np.maximum(0.0, (1.0 / a) * (1.0 - np.abs(u) / a))


def rectangular_rdd(y, x, c=C_CUTOFF, h=13.8):
    """
    方程 (21.4)：在 |X-c|<=h 上 OLS
        Y = b0 + b1*X + b3*(X-c)*D + theta*D + e
    返回 theta, se(HC), n, 全部系数。
    这等价于矩形核局部线性 RDD。
    """
    y = np.asarray(y, float)
    x = np.asarray(x, float)
    m = np.isfinite(y) & np.isfinite(x) & (np.abs(x - c) <= h)
    y, x = y[m], x[m]
    D = (x >= c).astype(float)
    Xmat = np.column_stack([np.ones(len(y)), x, (x - c) * D, D])
    b = pinv(Xmat.T @ Xmat) @ (Xmat.T @ y)
    e = y - Xmat @ b
    n, k = Xmat.shape
    # HC1 三明治协方差
    meat = Xmat.T @ (Xmat * (e ** 2)[:, None])
    bread = pinv(Xmat.T @ Xmat)
    V = (n / (n - k)) * bread @ meat @ bread
    se = np.sqrt(np.maximum(np.diag(V), 0.0))
    # theta 是 D 的系数，下标 3
    return float(b[3]), float(se[3]), n, b, se


def ll_side(y, x, c, h, side="left"):
    """
    单侧局部线性：在 cutoff 处估 m(c-) 或 m(c+)。
    Z_i = (1, X_i - c)，权为标准化三角核 K((X-c)/h)。
    """
    y = np.asarray(y, float)
    x = np.asarray(x, float)
    if side == "left":
        m = np.isfinite(y) & np.isfinite(x) & (x < c)
    else:
        m = np.isfinite(y) & np.isfinite(x) & (x >= c)
    y, x = y[m], x[m]
    w = triangular_normalized_kernel((x - c) / h)
    m2 = w > 0
    y, x, w = y[m2], x[m2], w[m2]
    if len(y) < 5:
        return np.nan, np.nan, 0
    Z = np.column_stack([np.ones(len(y)), x - c])
    XtWX = Z.T @ (Z * w[:, None])
    b = pinv(XtWX) @ (Z.T @ (w * y))
    e = y - Z @ b
    # 加权 HC meat: score_i = w_i * e_i * Z_i
    meat = np.zeros((2, 2))
    for i in range(len(y)):
        s = w[i] * e[i] * Z[i]
        meat += np.outer(s, s)
    V = pinv(XtWX) @ meat @ pinv(XtWX)
    return float(b[0]), float(np.sqrt(max(V[0, 0], 0.0))), len(y)


def rdd_ll(y, x, c=C_CUTOFF, h=8.0):
    """Sharp RDD: theta = m(c+) - m(c-)，两侧独立故 var 相加。"""
    m_minus, se_m, n0 = ll_side(y, x, c, h, side="left")
    m_plus, se_p, n1 = ll_side(y, x, c, h, side="right")
    theta = m_plus - m_minus
    se = np.sqrt(se_m ** 2 + se_p ** 2)
    return theta, se, m_plus, m_minus, n0, n1


## 载入数据

In [ ]:
df = load_lm2007()
x = df["povrate60"].values
print("n =", len(df), "cutoff c =", C_CUTOFF)
print("treated X>=c :", np.sum(x >= C_CUTOFF))


## 21.5 矩形核 (21.4)，h = 13.8, 7, 20

In [ ]:
y = df["mort_age59_related_postHS"].values
for h in [13.8, 7.0, 20.0]:
    th, se, n, b, _ = rectangular_rdd(y, x, h=h)
    print(f"h={h:5.1f}: theta={th:7.3f}  se={se:6.3f}  n={n}")
    print(f"         coeffs b0,b1,b3,theta = {np.round(b, 4)}")


## 21.6 基线 LL（标准化三角核），h = 8, 4, 12

$h=8$ 应接近 Table 21.1 的 $-1.51\ (0.71)$。

In [ ]:
for h in [8.0, 4.0, 12.0]:
    th, se, mp, mm, n0, n1 = rdd_ll(y, x, h=h)
    print(f"h={h:5.1f}: theta={th:7.3f}  se={se:6.3f}  m-={mm:.3f} m+={mp:.3f}  nL,nR={n0},{n1}")


## 21.7–21.9 安慰剂结果变量

In [ ]:
placebos = {
    "21.7 injury postHS (ages 5-9)": "mort_age59_injury_postHS",
    "21.8 related postHS (ages 25+)": "mort_age25plus_related_postHS",
    "21.9 related preHS (ages 5-9, 1959-64)": "mort_age59_related_preHS",
}
for label, col in placebos.items():
    y2 = df[col].values
    th8, se8, *_ = rdd_ll(y2, x, h=8.0)
    thr, ser, nr, *_ = rectangular_rdd(y2, x, h=13.8)
    print(label)
    print(f"  LL h=8:     theta={th8:7.3f}  se={se8:6.3f}")
    print(f"  Rect h=13.8: theta={thr:7.3f}  se={ser:6.3f}  n={nr}")
